# 실습 21. 비용 자료 분석

사회약학 연구방법 노트의 '실습 21 비용 자료 분석'에 나오는 셀을 차례로 모은 노트북입니다. 진행성 신세포암 1차 치료(신약 A 대 표준요법 B, 가상 자료) 환자 3,000명의 1년 의료비로 평균 비용의 차이와 부트스트랩 신뢰구간을 구하고, 감마 GLM으로 환자 구성의 차이를 보정합니다. 셀을 위에서부터 차례로 실행하세요. 자료는 사이트에서 바로 읽으므로 인터넷 연결이 필요합니다. 설명과 출력 읽는 법은 사이트의 실습 21 쪽에 있습니다.


## 가. 실습 데이터 준비

환자 한 명이 한 줄인 1년 의료비 자료(가상)를 불러와 군별로 요약하고 분포를 봅니다.

### 셀 1. 패키지와 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "rcc_cost.csv", storage_options=UA)
print(df.shape)
df.head()

### 셀 2. 치료군 변수 만들기, 결측값과 0원 확인

In [ ]:
df["trt"] = (df["arm"] == "A").astype(int)   # 신약 A = 1, B = 0
print(df["arm"].value_counts())
print("결측값:", df.isna().sum().sum(), "개")

items = ["cost_drug", "cost_op", "cost_inp", "cost"]
zero = (df[items] == 0).groupby(df["arm"]).mean() * 100
zero.round(1)

### 셀 3. 군별 요약통계

In [ ]:
print(df.groupby("arm")["cost"].describe().round(1).T)

df.groupby("arm")[items].agg(["mean", "std"]).round(1).T

### 셀 4. 비용 분포의 히스토그램

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, g in zip(axes, ["A", "B"]):
    y = df.loc[df["arm"] == g, "cost"]
    ax.hist(y, bins=np.arange(0, 10200, 200))
    ax.axvline(y.mean(), color="C1", label="mean")
    ax.axvline(y.median(), color="C1", ls="--", label="median")
    ax.set_title("Arm " + g)
    ax.set_xlabel("1-year cost (10,000 KRW)")
axes[0].set_ylabel("Number of patients")
axes[0].legend()
plt.tight_layout()
print("10,000만원을 넘어 그림 밖에 있는 환자:",
      (df["cost"] > 10000).groupby(df["arm"]).sum().to_dict())

### 셀 5. 평균, 중앙값, 총액

In [ ]:
a = df.loc[df["arm"] == "A", "cost"].to_numpy()   # A군 1,203명의 비용
b = df.loc[df["arm"] == "B", "cost"].to_numpy()   # B군 1,797명의 비용

print("B군 총액      :", round(b.sum()))
print("평균 x 인원   :", round(b.mean() * len(b)))
print("중앙값 x 인원 :", round(np.median(b) * len(b)))
print("평균보다 적게 쓴 환자(%):",
      round((b < b.mean()).mean() * 100, 1))
k = round(len(b) * 0.05)                          # 상위 5%는 90명
top = np.sort(b)[::-1][:k]
print("상위 5%가 쓴 몫(%)      :",
      round(top.sum() / b.sum() * 100, 1))

## 나. 평균 비용의 비교와 부트스트랩

평균 차이의 신뢰구간을 Welch t 검정과 부트스트랩으로 구하고, 중앙값·Mann-Whitney 검정·로그 변환이 답하는 질문과 견줍니다.

### 셀 6. 평균 차이와 Welch t 검정

In [ ]:
diff = a.mean() - b.mean()
tt = stats.ttest_ind(a, b, equal_var=False)       # Welch t 검정
ci = tt.confidence_interval()
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))

print("평균 차이(A - B):", round(diff, 1))
print("SE:", round(se, 1), " t =", round(tt.statistic, 2),
      " df =", round(tt.df, 1), " p =", tt.pvalue)
print("95% CI:", round(ci.low, 1), "~", round(ci.high, 1))

### 셀 7. A군을 한 번 다시 뽑아 보기

In [ ]:
rng = np.random.default_rng(21)
idx = rng.integers(0, len(a), len(a))    # 0~1202 가운데 1,203개
print(idx[:8])
print("한 번 이상 뽑힌 환자:", len(np.unique(idx)), "명")
print("원래 A군의 평균     :", round(a.mean(), 1))
print("다시 뽑은 A군의 평균:", round(a[idx].mean(), 1))

### 셀 8. 군 안에서 9,999번 다시 뽑기

In [ ]:
B = 9999                                     # 반복 횟수
rng = np.random.default_rng(21)              # 난수를 처음부터 다시
ia = rng.integers(0, len(a), (B, len(a)))    # A군 번호표
ib = rng.integers(0, len(b), (B, len(b)))    # B군 번호표
print(ia.shape, ib.shape)

boot = a[ia].mean(axis=1) - b[ib].mean(axis=1)
print(boot.shape, boot[:4].round(1))

lo, hi = np.percentile(boot, [2.5, 97.5])
print("95% CI (percentile):", round(lo, 1), "~", round(hi, 1))
print("bootstrap SE:", round(boot.std(ddof=1), 1))

### 셀 9. 부트스트랩 분포 그림

In [ ]:
plt.figure(figsize=(6.4, 3.4))
plt.hist(boot, bins=50)
plt.axvline(diff, color="C1", label="observed difference")
plt.axvline(lo, color="k", ls="--", label="2.5% and 97.5%")
plt.axvline(hi, color="k", ls="--")
plt.xlabel("Bootstrap mean difference, A - B (10,000 KRW)")
plt.ylabel("Number of resamples")
plt.legend()
plt.tight_layout()
print("0 이하인 값:", (boot <= 0).sum(), "개")

### 셀 10. scipy 함수로 확인하고 BCa 구간 구하기

In [ ]:
def mean_diff(x, y, axis=-1):
    return x.mean(axis=axis) - y.mean(axis=axis)

for method in ["percentile", "BCa"]:
    res = stats.bootstrap((a, b), mean_diff, n_resamples=9999,
                          method=method,
                          random_state=np.random.default_rng(21))
    bci = res.confidence_interval
    print(method, round(bci.low, 1), round(bci.high, 1),
          round(res.standard_error, 1))

### 셀 11. 중앙값 차이와 Mann-Whitney 검정

In [ ]:
med_diff = np.median(a) - np.median(b)
mw = stats.mannwhitneyu(a, b)
print("중앙값 A, B:", np.median(a), np.median(b),
      " 차이:", round(med_diff, 1))
print("Mann-Whitney U =", mw.statistic, " p =", mw.pvalue)
print("A군 환자의 비용이 더 클 확률:",
      round(mw.statistic / (len(a) * len(b)), 3))

print("총액 차이, 평균 차이 x 1,203명  :", round(diff * len(a)))
print("총액 차이, 중앙값 차이 x 1,203명:", round(med_diff * len(a)))

### 셀 12. 고액 환자의 비용을 키워 보는 실험

In [ ]:
b2 = b.copy()
top = b2 >= np.percentile(b2, 95)       # B군에서 비용이 큰 5%
b2[top] = b2[top] * 3                   # 이 환자들의 비용만 3배로

print("바꾼 환자:", top.sum(), "명")
print("평균 차이(A - B):", round(a.mean() - b2.mean(), 1))
print("중앙값 차이      :", round(np.median(a) - np.median(b2), 1))
mw2 = stats.mannwhitneyu(a, b2)
print("A군이 더 클 확률 :",
      round(mw2.statistic / (len(a) * len(b2)), 3),
      " p =", mw2.pvalue)

### 셀 13. 로그 변환, 기하평균, smearing 계수

In [ ]:
la, lb = np.log(a), np.log(b)
gm_a, gm_b = np.exp(la.mean()), np.exp(lb.mean())
print("로그 비용의 SD A, B:", round(la.std(ddof=1), 2),
      round(lb.std(ddof=1), 2))
print("기하평균 A, B:", round(gm_a, 1), round(gm_b, 1))

tl = stats.ttest_ind(la, lb, equal_var=False)
cl = tl.confidence_interval()
print("기하평균의 비:", round(gm_a / gm_b, 3),
      np.exp([cl.low, cl.high]).round(3))
print("산술평균의 비:", round(a.mean() / b.mean(), 3))

k_a = np.exp(la - la.mean()).mean()      # smearing 계수
k_b = np.exp(lb - lb.mean()).mean()
print("smearing 계수 A, B:", round(k_a, 3), round(k_b, 3))
print("기하평균 x 계수   :",
      round(gm_a * k_a, 1), round(gm_b * k_b, 1))

## 다. 비용의 회귀분석

두 군의 환자 구성 차이를 선형회귀, 로그 변환 회귀, 감마 GLM으로 보정합니다. 셀 20은 모형을 1,000번 다시 적합하므로 1–2분 걸립니다.

### 셀 14. 두 군의 기저 특성

In [ ]:
covs = ["age", "male", "cci", "stage4"]
df.groupby("arm")[covs].agg(["mean", "std"]).round(2).T

### 셀 15. 원래 비용의 선형회귀 (강건 표준오차)

In [ ]:
f = "cost ~ trt + age + male + cci + stage4"
ols = smf.ols(f, data=df).fit(cov_type="HC3")
print(ols.summary().tables[1])

### 셀 16. 로그 변환 회귀와 되돌리기

In [ ]:
logm = smf.ols("np.log(cost) ~ trt + age + male + cci + stage4",
               data=df).fit()
print("exp(b):", np.exp(logm.params["trt"]).round(3),
      np.exp(logm.conf_int().loc["trt"]).round(3).tolist())

dA, dB = df.assign(trt=1), df.assign(trt=0)   # 모두 A, 모두 B
nA = np.exp(logm.predict(dA)).mean()
nB = np.exp(logm.predict(dB)).mean()
print("그대로 되돌림  :", round(nA, 1), round(nB, 1),
      round(nA - nB, 1))

s = np.exp(logm.resid).mean()                 # smearing 계수 하나
print("smearing 계수  :", round(s, 3))
print("계수 하나로 보정:", round(nA * s, 1), round(nB * s, 1),
      round((nA - nB) * s, 1))

sA = np.exp(logm.resid[df["trt"] == 1]).mean()
sB = np.exp(logm.resid[df["trt"] == 0]).mean()
print("군별 계수 A, B :", round(sA, 3), round(sB, 3))
print("군별 계수로 보정:", round(nA * sA, 1), round(nB * sB, 1),
      round(nA * sA - nB * sB, 1))

### 셀 17. 감마 GLM (로그 연결)

In [ ]:
fam = sm.families.Gamma(link=sm.families.links.Log())
glm = smf.glm(f, data=df, family=fam).fit()
print(glm.summary())

### 셀 18. 계수를 비용 비로 바꾸기

In [ ]:
ratio = pd.DataFrame({"cost_ratio": np.exp(glm.params),
                      "lower": np.exp(glm.conf_int()[0]),
                      "upper": np.exp(glm.conf_int()[1]),
                      "p": glm.pvalues})
ratio.round(3)

### 셀 19. 모두 A일 때와 모두 B일 때의 예측 평균

In [ ]:
pA = glm.predict(df.assign(trt=1))   # 모두 A를 썼다고 놓은 예측
pB = glm.predict(df.assign(trt=0))   # 모두 B를 썼다고 놓은 예측
print("모두 A, 모두 B, 차이:", round(pA.mean(), 1),
      round(pB.mean(), 1), round(pA.mean() - pB.mean(), 1))
print("비:", round(pA.mean() / pB.mean(), 3))

d_i = pA - pB                        # 환자 한 명 한 명의 예측 차이
print("환자별 차이의 범위:", round(d_i.min(), 1), "~",
      round(d_i.max(), 1))

two = pd.DataFrame({"trt": [1, 0, 1, 0], "age": [64, 64, 75, 75],
                    "male": 1, "cci": [1, 1, 4, 4], "stage4": 1})
print(glm.predict(two).round(1).tolist())

### 셀 20. 표준화한 차이의 부트스트랩 신뢰구간

In [ ]:
rng = np.random.default_rng(2101)
n = len(df)
R = 1000                                  # 반복 횟수
bs = np.empty((R, 3))                     # 1,000행 x 3열의 빈 상자
for i in range(R):
    ix = rng.integers(0, n, n)            # 환자 3,000명을 다시 뽑음
    d = df.iloc[ix]
    m = smf.glm(f, data=d, family=fam).fit()
    mA = m.predict(d.assign(trt=1)).mean()
    mB = m.predict(d.assign(trt=0)).mean()
    bs[i] = [mA, mB, mA - mB]

ci3 = np.percentile(bs, [2.5, 97.5], axis=0).round(1)
print("모두 A :", ci3[:, 0])
print("모두 B :", ci3[:, 1])
print("차이   :", ci3[:, 2], " SE:", round(bs[:, 2].std(ddof=1), 1))

### 셀 21. 분포와 평균 구조 점검

In [ ]:
chk = pd.DataFrame({"cost": df["cost"], "pred": glm.predict(df)})
chk["res2"] = (chk["cost"] - chk["pred"]) ** 2
chk["ln_pred"] = np.log(chk["pred"])
park = smf.glm("res2 ~ ln_pred", data=chk,
               family=fam).fit(cov_type="HC0")
print("기울기:", park.params["ln_pred"].round(2),
      park.conf_int().loc["ln_pred"].round(2).tolist())

chk["tenth"] = pd.qcut(chk["pred"], 10, labels=False) + 1
chk.groupby("tenth")[["cost", "pred"]].mean().round(0)

### 셀 22. 방법별 결과 모으기

In [ ]:
res_tab = pd.DataFrame(
    {"all_A": [a.mean(), ols.predict(dA).mean(), nA, nA * s,
               nA * sA, pA.mean()],
     "all_B": [b.mean(), ols.predict(dB).mean(), nB, nB * s,
               nB * sB, pB.mean()]},
    index=["unadjusted", "OLS", "log OLS, naive", "log OLS, smearing",
           "log OLS, smearing by arm", "gamma GLM"])
res_tab["diff"] = res_tab["all_A"] - res_tab["all_B"]
res_tab.round(0)

## 과제 정답

**과제 1.** 약제비(`cost_drug`), 외래 진료비(`cost_op`), 입원 진료비(`cost_inp`), 총비용(`cost`) 각각에 대해 평균 차이(A − B)와 부트스트랩 백분위수 95% 신뢰구간(9,999회, seed 21)을 구합니다. 먼저 스스로 풀어 본 뒤 아래 셀을 실행하세요.

### 셀 23. 과제 1 정답. 항목별 평균 차이와 부트스트랩 구간

In [ ]:
def boot_diff(col, B=9999, seed=21):
    x = df.loc[df["arm"] == "A", col].to_numpy()
    y = df.loc[df["arm"] == "B", col].to_numpy()
    rng = np.random.default_rng(seed)
    ix = rng.integers(0, len(x), (B, len(x)))
    iy = rng.integers(0, len(y), (B, len(y)))
    bt = x[ix].mean(axis=1) - y[iy].mean(axis=1)
    lo, hi = np.percentile(bt, [2.5, 97.5])
    return [x.mean(), y.mean(), x.mean() - y.mean(), lo, hi]

hw1 = pd.DataFrame([boot_diff(col) for col in items], index=items,
                   columns=["mean_A", "mean_B", "diff",
                            "lower", "upper"])
hw1.round(0)

**과제 2.** 반복 횟수를 200, 1,000, 9,999로 바꾸고 seed를 1부터 10까지 바꿔 가며 총비용 평균 차이의 백분위수 구간을 구해, 구간의 하한과 상한이 seed에 따라 얼마나 달라지는지 비교합니다.

### 셀 24. 과제 2 정답. 반복 횟수와 구간의 흔들림

In [ ]:
rows = []
for B_ in [200, 1000, 9999]:
    lows, highs = [], []
    for seed in range(1, 11):                 # seed 1~10
        rng = np.random.default_rng(seed)
        ia_ = rng.integers(0, len(a), (B_, len(a)))
        ib_ = rng.integers(0, len(b), (B_, len(b)))
        bt = a[ia_].mean(axis=1) - b[ib_].mean(axis=1)
        l_, h_ = np.percentile(bt, [2.5, 97.5])
        lows.append(l_)
        highs.append(h_)
    rows.append([B_, min(lows), max(lows), min(highs), max(highs)])

hw2 = pd.DataFrame(rows, columns=["B", "lower_min", "lower_max",
                                  "upper_min", "upper_max"])
hw2["lower_range"] = hw2["lower_max"] - hw2["lower_min"]
hw2["upper_range"] = hw2["upper_max"] - hw2["upper_min"]
hw2.round(1)

**과제 3.** 입원 진료비(`cost_inp`)는 38.8%가 0원입니다. 이단계 모형(1부 로지스틱 회귀, 2부 감마 GLM)으로 모두 A일 때와 모두 B일 때의 1인당 입원비와 그 차이를 구합니다.

### 셀 25. 과제 3 정답. 입원비의 이단계 모형

In [ ]:
df["anyinp"] = (df["cost_inp"] > 0).astype(int)  # 입원비가 생겼으면 1
rhs = "trt + age + male + cci + stage4"
part1 = smf.logit("anyinp ~ " + rhs, data=df).fit(disp=0)
pos = df[df["cost_inp"] > 0]                     # 입원비가 있는 환자만
part2 = smf.glm("cost_inp ~ " + rhs, data=pos, family=fam).fit()
print("입원비가 0보다 큰 환자:", len(pos), "명")
print("1부 오즈비:", np.exp(part1.params["trt"]).round(3),
      np.exp(part1.conf_int().loc["trt"]).round(2).tolist())
print("2부 비용 비:", np.exp(part2.params["trt"]).round(3),
      np.exp(part2.conf_int().loc["trt"]).round(2).tolist())

dA, dB = df.assign(trt=1), df.assign(trt=0)
eA = (part1.predict(dA) * part2.predict(dA)).mean()
eB = (part1.predict(dB) * part2.predict(dB)).mean()
print("모두 A, 모두 B, 차이:", round(eA, 1), round(eB, 1),
      round(eA - eB, 1))